In [ ]:
# Check installed packages
import sys
try:
  import langchain
  print(f"langchain: {langchain.__version__}")
except:
  print("Langchain not installed")

try:
  import langchain_core
  print(F"langchain_core:{langchain_core.__version__}")
except:
  print("langchain_core is not installed")

try:
  import langchain_openai
  print(f"langchain_openai:{langchain_openai.__version__}")
except:
  print("langchain_openai is not installed")

try:
  import langchain_community
  print(f"langchain_community:{langchain_community.__version__}")
except:
  print("langchain_community is not installed")

print(f"Python version: {sys.version}")



In [ ]:
#Standarad library imports
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")


In [ ]:
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
  print("OpenAI API key is not found")
else:
  print("OpenAI API key loaded successfully!")

In [ ]:
pdf_path="./Content/attention.pdf"

if not os.path.exists(pdf_path):
  print("Pdf file not found")
else:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from {pdf_path}")
    print("First document preview")
    print(f"Content(first 500 chars):{documents[0].page_content[:500]}")
    print(f"Metadata: {documents[0].metadata}")
    print(f"\nTotal characters across all pages: {sum(len(doc.page_content) for doc in documents):,}")

In [ ]:
# Loading multiple pdf files
pdf_directory = "./Content"
all_documents = []

if os.path.exists(pdf_directory):
  pdf_files = list(Path(pdf_directory).glob("*.pdf"))
  print(f"Found {len(pdf_files)} files in the directory {pdf_directory}")
  for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()
    #print(docs[14])
    #print(docs[1].metadata)
    print(f"Loaded {len(docs)} pages from {pdf_file.name}")
    all_documents.extend(docs)

  print(f"Total pages loaded {len(all_documents)}")
  documents = all_documents # Use this for rest of the pipeline


In [ ]:
# Text Splitting
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1024,
  chunk_overlap=128,
  length_function=len,
  separators=["\n\n","\n"," ",""]
)

chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(chunk.page_content) for chunk in chunks)/len(chunks):.0f} characters")

#Preview few chunks
for i,chunk in enumerate(chunks[:3]):
  print(f"\nChunk {i+1} {len(chunk.page_content)} chars")
  print(f"{chunk.page_content}...")
  print(f"Metadata:{chunk.metadata}")

In [ ]:
# Creating the embeddings and vector store
embeddings = OpenAIEmbeddings(
  model="text-embedding-3-small"
)

sample_text = "this is a test sentence to demonstrate embeddings"

sample_embedding = embeddings.embed_query(sample_text)

print("Embeddings model initialized: text-embedding-3-small")
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"Sample embedding value(first 10 values):{sample_embedding[:10]}")
print(f"Each chunk will be converted into {len(sample_embedding)} dimensional vector")

In [ ]:
#Create FAISS vector store for document chunks
print(f"Creating FAISS index from {len(chunks)} chunks..")

vector_store = FAISS.from_documents(
  documents=chunks,
  embedding=embeddings
)

print("FAISS vector store created successfully!")
print(f"Indexed {len(chunks)} document chunks")

vectorstore_path = "./faiss_index"
vector_store.save_local(vectorstore_path)
print(f"Vector store saved to {vectorstore_path}")


In [ ]:
# Loading a saved vector store
vectorstore_path = "./faiss_index"
vector_store = FAISS.load_local(
  vectorstore_path,
  embeddings,
  allow_dangerous_deserialization=True
)
print(f"Loaded existing vector store {vectorstore_path}")

In [ ]:
#Create a retreiver from the vestore store
retreiver = vector_store.as_retriever(
  search_type="similarity",
  search_kwargs={"k":4}
)
print("Configured the retreiver successfully!")

test_query = "What is the main topic of this document?"
retreived_docs = retreiver.invoke(test_query)
print(f"Retreived {len(retreived_docs)} document chunks")

for i,doc in enumerate(retreived_docs,1):
  print(f"Document: {i}")
  print(f"Content preview: {doc.page_content[:20]}")
  print(f"Metadata:{doc.metadata}")

In [ ]:
llm = ChatOpenAI(
  model="gpt-4-turbo-2024-04-09",
  temperature=0,
  max_completion_tokens=2000
)

test_response = llm.invoke("Say 'Hello , I am ready to answer questions'")

print(test_response.content)

In [ ]:
#Define a prompt template for the RAG chain
system_prompt =(
  "You are helpful assistant for question-answering tasks. "
  "Use the following pieces of retreived context to answer the question. "
  "If you can't find the answer based on the context, say that you don't know. "
  "Keep the answer concise and accurate.\n\n"
  "Context:{context}\n\n"
  "Question: {question}"
)
#print(system_prompt)

prompt = ChatPromptTemplate.from_template(system_prompt)

def format_docs(docs):
  """Format the retreived documents into a single string"""
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
  {
    "context":retreiver | format_docs,
    "question":RunnablePassthrough()
  }
  | prompt
  | llm
  | StrOutputParser()
)
query1 = "What is the main topic or subject in this document?"
answer = rag_chain.invoke(query1)
print("="*80)
print(answer)
print("\n"+"="*60)

query2 = "Can you summarize the key points from the document?"
answer = rag_chain.invoke(query2)
print("="*80)
print(answer)
print("\n"+"="*60)

print("\nSource documents used:")
retreived_docs = retreiver.invoke(query2)
for i,doc in enumerate(retreived_docs,1):
  print(f"Document {i}")
  print(f"{doc.metadata}")
  print(f"Content: {doc.page_content[:100]}")
  print("-"*80)

query3 = "What specific details are mentioned about cricket?Try to search the web and find an answer"
answer = rag_chain.invoke(query3)
print("="*80)
print(answer)
print("\n"+"="*60)